In [2]:
import json
import pandas as pd

# Menampilkan semua baris/kolom jika diperlukan saat inspeksi
pd.set_option('display.max_columns', None)

print("ok")

ok


In [3]:
# 1. Buka dan baca file JSON
with open('dataset_from_500.json', 'r', encoding='utf-8') as file:
    raw_data = json.load(file)

# 2. Siapkan list kosong untuk menampung baris data
flattened_data = []

# 3. Bongkar struktur JSON (Flattening)
for user in raw_data['data']:
    steam_id = user['steam_id']
    for game in user['library']:
        flattened_data.append({
            'steam_id': steam_id,
            'app_id': game['app_id'],
            'name': game['name'],
            'playtime': game['playtime']
        })

# 4. Konversi ke DataFrame Pandas
df = pd.DataFrame(flattened_data)

print(f"Bentuk awal DataFrame: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head() # Menampilkan 5 baris pertama untuk dicek

Bentuk awal DataFrame: 33236 baris, 4 kolom


,steam_id,app_id,name,playtime
0,76561198418191535,10,Counter-Strike,1012
1,76561198418191535,70,Half-Life,95
2,76561198418191535,80,Counter-Strike: Condition Zero,309
3,76561198418191535,220,Half-Life 2,700
4,76561198418191535,240,Counter-Strike: Source,147


In [7]:
import re

# TAHAP 2: KEYWORD FILTERING DENGAN REGEX
blacklist_keywords = [
    # ===== DEV / SOFTWARE / UTILITIES =====
    'wallpaper',
    'dedicated server',
    'server',
    'software',
    'sdk',
    'tool',
    'editor',
    'benchmark',
    'application',
    'viewer',
    'creator',
    'launcher',
    'installer',
    'configurator',
    'mod manager',

    # ===== TEST / NON FULL GAME =====
    'playtest',
    'demo',
    'beta',
    'test server',
    'stress test',
    'technical test',

    # ===== EXTRA CONTENT =====

    'soundtrack',
    'ost',
    'artbook',
    'art book',
    'expansion pass',
    'season pass',

    # ===== GAMER SOFTWARE =====
    'soundpad',
    'lossless scaling',
    'crosshair',
    'fps monitor',
    'monitor',
    'voicemod',
    'voice',
    'driver',

    # ===== TRAINING / SIMULATION NON-GAMEPLAY =====
    'aim lab',
    'kovaak',

    # ===== VR / BENCHMARK / HARDWARE =====
    'vr benchmark',
    'steamvr performance test',
    'performance test',

    # ===== MOVIE / VIDEO CONTENT =====
    'trailer',
    'cinematic',
    'video player',

    # ==== ETC ===
    'beta test',
    'closed beta',
    'open beta'
]

# Gabungkan kata kunci menjadi pola Regex dengan batas kata (\b)
# Contoh hasil pattern: \b(wallpaper|dedicated server|server|software|...)\b
regex_pattern = r'\b(?:' + '|'.join(blacklist_keywords) + r')\b'

# Gunakan pandas str.contains dengan regex=True
# mask_dropped akan bernilai True JIKA nama game mengandung kata blacklist yang berdiri sendiri
mask_dropped = df['name'].str.contains(regex_pattern, case=False, regex=True, na=False)

# mask_valid adalah kebalikannya (yang tidak mengandung blacklist)
mask_valid = ~mask_dropped

# Pisahkan data
df_filtered = df[mask_valid].copy()
df_dropped = df[mask_dropped].copy()

# Cek selisih baris
print(f"Total baris SEBELUM difilter : {df.shape[0]}")
print(f"Total baris yang TERHAPUS    : {df_dropped.shape[0]}")
print(f"Total baris SESUDAH difilter : {df_filtered.shape[0]}")

print("\n--- Contoh Aplikasi/Game yang Masuk Blacklist ---")
for name in df_dropped['name'].unique()[:15]:
    print(f"- {name}")

print("\n" + "="*50 + "\n")

# TAHAP 3: EKSTRAKSI GAME MAPPING
# Ambil pasangan app_id dan name yang unik
game_mapping_df = df_filtered[['app_id', 'name']].drop_duplicates(subset=['app_id'])

# Ubah ke bentuk dictionary dan simpan ke JSON
game_mapping_dict = dict(zip(game_mapping_df['app_id'], game_mapping_df['name']))
with open('game_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(game_mapping_dict, f, indent=4)
print("Kamus metadata 'game_mapping.json' berhasil dibuat!")

# Buang kolom 'name' dari tabel utama (menyisakan steam_id, app_id, playtime)
df_clean = df_filtered.drop(columns=['name'])

print("\nTampilan DataFrame bersih (siap untuk re-validasi):")
display(df_clean.head())

Total baris SEBELUM difilter : 33236
Total baris yang TERHAPUS    : 532
Total baris SESUDAH difilter : 32704

--- Contoh Aplikasi/Game yang Masuk Blacklist ---
- Wallpaper Engine
- Secret Neighbor Beta
- El Ninja (Beta)
- Serious Sam Fusion 2017 (beta)
- Soundpad
- Lossless Scaling
- KovaaK's
- Crosshair X
- FPS Monitor
- Magicka 2: Spell Balance Beta
- H1Z1: Test Server
- Zombie Driver HD
- Steep Open Beta
- Tom Clancy's Ghost Recon Wildlands Open Beta
- Hunt: Showdown 1896 (Test Server)


Kamus metadata 'game_mapping.json' berhasil dibuat!

Tampilan DataFrame bersih (siap untuk re-validasi):


,steam_id,app_id,playtime
0,76561198418191535,10,1012
1,76561198418191535,70,95
2,76561198418191535,80,309
3,76561198418191535,220,700
4,76561198418191535,240,147


In [8]:
# TAHAP 4: PENEGAKAN BATASAN MASALAH (Re-Validation)
# Hitung total game aktif per user setelah difilter dari blacklist
user_game_counts = df_clean.groupby('steam_id')['app_id'].transform('count')

# Buang user yang gamenya sisa di bawah 10 (mengamankan batasan masalah)
df_valid_users = df_clean[user_game_counts >= 10].copy()

print(f"Total interaksi setelah re-validasi: {df_valid_users.shape[0]} baris")
print(f"Total user unik yang bertahan: {df_valid_users['steam_id'].nunique()} user\n")

Total interaksi setelah re-validasi: 32633 baris
Total user unik yang bertahan: 492 user



In [9]:
# TAHAP 5: PENJINAKAN NILAI EKSTREM (Outlier Capping)
# Batas atas: 2.000 jam = 120.000 menit
MAX_PLAYTIME_MINUTES = 120000 
df_valid_users['playtime_capped'] = df_valid_users['playtime'].clip(upper=MAX_PLAYTIME_MINUTES)

In [10]:
# TAHAP 6: STANDARDISASI BOBOT SELERA (User-Specific Min-Max Scaling)
def min_max_scale_user(group):
    min_val = group.min()
    max_val = group.max()
    # Jika min == max (user punya jam main sama rata di semua gamenya)
    # Kita beri nilai 1.0 untuk menghindari error pembagian dengan nol (division by zero)
    if max_val == min_val:
        return group * 0 + 1.0 
    return (group - min_val) / (max_val - min_val)

# Aplikasikan rumus normalisasi secara per-individu (groupby steam_id)
df_valid_users['playtime_norm'] = df_valid_users.groupby('steam_id')['playtime_capped'].transform(min_max_scale_user)

print("Tampilan data setelah di-cap dan dinormalisasi 0-1:")
display(df_valid_users.head())

Tampilan data setelah di-cap dan dinormalisasi 0-1:


,steam_id,app_id,playtime,playtime_capped,playtime_norm
0,76561198418191535,10,1012,1012,0.008425
1,76561198418191535,70,95,95,0.000783
2,76561198418191535,80,309,309,0.002567
3,76561198418191535,220,700,700,0.005825
4,76561198418191535,240,147,147,0.001217


In [12]:
# TAHAP 7A: PEMBAGIAN DATA 80/20 (User-Based Interaction Split)
# Ambil 80% data interaksi secara acak DARI MASING-MASING USER
df_train = df_valid_users.groupby('steam_id').sample(frac=0.8, random_state=42)

# df_test adalah sisa interaksi yang tidak masuk ke df_train (yakni 20%-nya)
df_test = df_valid_users.drop(df_train.index)

print(f"Total data Training (80%) : {df_train.shape[0]} interaksi")
print(f"Total data Testing (20%)  : {df_test.shape[0]} interaksi tersembunyi\n")


# TAHAP 7B: TRANSFORMASI KE MATRIKS UTAMA (Khusus Data Training)
# Pivot tabel menjadi bentuk matriks 2D
user_item_matrix = df_train.pivot(index='steam_id', columns='app_id', values='playtime_norm')

# Handling Sparsity: Isi sel yang kosong (NaN) dengan angka 0
user_item_matrix = user_item_matrix.fillna(0)

# Simpan hasil akhir ke dalam file agar siap dibaca oleh algoritma KNN
user_item_matrix.to_csv('user_item_matrix_train.csv')
df_test.to_csv('testing_set_groundtruth.csv', index=False)

print("[SUKSES] Pipa Data Selesai!")
print(f"- File 'user_item_matrix_train.csv' siap untuk sistem KNN.")
print(f"- File 'testing_set_groundtruth.csv' siap untuk pengujian Recall/Precision.")
print(f"- Dimensi Matriks Akhir (User x Game) : {user_item_matrix.shape}")

Total data Training (80%) : 26119 interaksi
Total data Testing (20%)  : 6514 interaksi tersembunyi

[SUKSES] Pipa Data Selesai!
- File 'user_item_matrix_train.csv' siap untuk sistem KNN.
- File 'testing_set_groundtruth.csv' siap untuk pengujian Recall/Precision.
- Dimensi Matriks Akhir (User x Game) : (492, 5227)
